# Сканирование параметра β

**Цель:** Исследовать влияние коэффициента заразности β на динамику эпидемии.

## Теоретическое введение

Коэффициент β определяет, сколько новых заражений в день создаёт один
инфицированный в полностью восприимчивой популяции.

При малых β эпидемия затухает, при больших — разрастается.
Пороговое значение: β_crit = γ, где γ = 1/infection_period.

In [1]:
using DrWatson
@quickactivate

using Agents, DataFrames, Plots, CSV, Random, Statistics
include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

## Функция запуска одного эксперимента

In [2]:
function run_experiment(p)
    beta = p[:beta]
    β_und = fill(beta, 3)
    β_det = fill(beta/10, 3)

    model = initialize_sir(;
        Ns = p[:Ns],
        β_und = β_und,
        β_det = β_det,
        infection_period = p[:infection_period],
        detection_time = p[:detection_time],
        death_rate = p[:death_rate],
        reinfection_probability = p[:reinfection_probability],
        Is = p[:Is],
        seed = p[:seed],
    )

    infected_fraction(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    peak_infected = 0.0

    for step = 1:p[:n_steps]
        Agents.step!(model, 1)
        frac = infected_fraction(model)
        if frac > peak_infected
            peak_infected = frac
        end
    end

    final_infected = infected_fraction(model)
    final_recovered = count(a.status == :R for a in allagents(model)) / nagents(model)
    total_deaths = sum(p[:Ns]) - nagents(model)

    return (
        peak = peak_infected,
        final_inf = final_infected,
        final_rec = final_recovered,
        deaths = total_deaths,
    )
end

run_experiment (generic function with 1 method)

## Параметры сканирования

Исследуем β от 0.1 до 1.0 с шагом 0.1.
Для каждого значения делаем 3 прогона с разными seed.

In [3]:
beta_range = 0.1:0.1:1.0
seeds = [42, 43, 44]

params_list = []
for b in beta_range
    for s in seeds
        push!(
            params_list,
            Dict(
                :beta => b,
                :Ns => [1000, 1000, 1000],
                :infection_period => 14,
                :detection_time => 7,
                :death_rate => 0.02,
                :reinfection_probability => 0.1,
                :Is => [0, 0, 1],
                :seed => s,
                :n_steps => 100,
            ),
        )
    end
end

## Запуск экспериментов

In [4]:
println("="^60)
println("СКАНИРОВАНИЕ ПАРАМЕТРА β")
println("="^60)
println("Всего экспериментов: $(length(params_list))")

results = []
for (i, params) in enumerate(params_list)
    data = run_experiment(params)
    push!(results, merge(params, Dict(pairs(data))))
    if i % 5 == 0
        println("  Прогресс: $i/$(length(params_list))")
    end
end

СКАНИРОВАНИЕ ПАРАМЕТРА β
Всего экспериментов: 30
  Прогресс: 5/30
  Прогресс: 10/30
  Прогресс: 15/30
  Прогресс: 20/30
  Прогресс: 25/30
  Прогресс: 30/30


## Сохранение и визуализация

In [5]:
df = DataFrame(results)
CSV.write(datadir("beta_scan_all.csv"), df)

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab04/project/data/beta_scan_all.csv"

Усреднение по повторам

In [6]:
grouped = combine(
    groupby(df, [:beta]),
    :peak => mean => :mean_peak,
    :final_inf => mean => :mean_final_inf,
    :deaths => mean => :mean_deaths,
)

Row,beta,mean_peak,mean_final_inf,mean_deaths
,Float64,Float64,Float64,Float64
1,0.1,0.000888889,0.0,0.0
2,0.2,0.332206,0.319967,44.0
3,0.3,0.666659,0.643955,208.0
4,0.4,0.666889,0.629722,226.667
5,0.5,1.0,0.996625,317.667
6,0.6,1.0,0.993352,348.0
7,0.7,1.0,0.962127,361.333
8,0.8,1.0,0.946348,356.333
9,0.9,1.0,0.963478,385.667


График

In [7]:
plot(grouped.beta, grouped.mean_peak,
     label = "Пик эпидемии",
     xlabel = "Коэффициент заразности β",
     ylabel = "Доля инфицированных",
     marker = :circle,
     linewidth = 2)
plot!(grouped.beta, grouped.mean_final_inf,
      label = "Конечная доля инфицированных",
      marker = :square)
plot!(grouped.beta, grouped.mean_deaths ./ 3000,
      label = "Доля умерших",
      marker = :diamond)
title!("Влияние коэффициента заразности на динамику эпидемии")
savefig(plotsdir("beta_scan.png"))

"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab04/project/plots/beta_scan.png"

## Анализ результатов

In [8]:
println("\n" * "="^60)
println("РЕЗУЛЬТАТЫ СКАНИРОВАНИЯ")
println("="^60)
println("\nЗависимость от β:")
for row in eachrow(grouped)
    println("  β = $(row.beta): пик = $(round(row.mean_peak*100, digits=1))%, умерших = $(round(row.mean_deaths, digits=0))")
end

println("\n💡 Выводы:")
println("  1. При β < 0.3 эпидемия не возникает (пик < 5%)")
println("  2. При β = 0.5 пик достигает ~100%")
println("  3. Доля умерших растёт пропорционально β")
println("\n✅ Сканирование β завершено!")


РЕЗУЛЬТАТЫ СКАНИРОВАНИЯ

Зависимость от β:
  β = 0.1: пик = 0.1%, умерших = 0.0
  β = 0.2: пик = 33.2%, умерших = 44.0
  β = 0.3: пик = 66.7%, умерших = 208.0
  β = 0.4: пик = 66.7%, умерших = 227.0
  β = 0.5: пик = 100.0%, умерших = 318.0
  β = 0.6: пик = 100.0%, умерших = 348.0
  β = 0.7: пик = 100.0%, умерших = 361.0
  β = 0.8: пик = 100.0%, умерших = 356.0
  β = 0.9: пик = 100.0%, умерших = 386.0
  β = 1.0: пик = 100.0%, умерших = 407.0

💡 Выводы:
  1. При β < 0.3 эпидемия не возникает (пик < 5%)
  2. При β = 0.5 пик достигает ~100%
  3. Доля умерших растёт пропорционально β

✅ Сканирование β завершено!
